# LLM-Based Narrative Structure Extraction for DMPBridge

## Objective

The goal of this experiment is to evaluate whether a Large Language Model (Llama 3.1 8B via Ollama) can improve the identification of narrative structure within Data Management Plans (DMPs).

Rather than generating the complete RDA Common Standard JSON directly, the LLM is used to detect and label the narrative components of a DMP, including document titles, section headings, subsection/question headings, and narrative content. The resulting structured blocks are then converted into a DMPTool-compatible narrative JSON using the existing DMPBridge narrative JSON builder.

---

## Workflow

```text
PDF
↓
PDFPlumber Extraction
↓
Markdown Text
↓
Llama 3.1 8B (Ollama)
↓
Structured Blocks
↓
Narrative JSON Builder
↓
Final Narrative JSON
```

---

## Input

The input to this experiment is the markdown representation of a DMP produced by the PDF extraction pipeline. The markdown file preserves the textual content extracted from the original PDF and serves as the source text for LLM-based structure detection.

---

## Structured Block Generation

The LLM analyzes the DMP text and converts it into a sequence of labeled blocks.

### Supported Labels

| Label | Description |
|---------|-------------|
| document_title | Main title of the DMP |
| section | Major DMP section heading |
| subsection | Question or subsection heading |
| content | Narrative answer text |

### Example

```json
{
  "label": "section",
  "text": "Types of Data"
}
```

The objective is to preserve the original wording while improving structural detection compared with rule-based approaches.

---

## Narrative JSON Construction

The structured blocks generated by the LLM are passed to the existing DMPBridge narrative JSON builder.

The builder converts the blocks into the DMPTool narrative structure:

- narrative.template.title
- narrative.template.section[]
- question[]
- answer[]

This approach preserves compatibility with the current DMPBridge architecture while allowing the LLM to improve section and question detection.

---

## Evaluation Criteria

### Structure Detection

- Correct identification of document titles
- Correct identification of section headings
- Correct identification of subsection/question headings
- Correct assignment of narrative text to the appropriate section

### Content Preservation

- No missing narrative content
- No hallucinated content
- Original wording preserved whenever possible
- Correct ordering of sections and answers

### JSON Quality

- Valid JSON generated
- Compatible with DMPBridge narrative schema
- Successfully converted into final narrative JSON
- No schema validation errors

---

## Expected Output

### Structured Blocks

```json
[
  {
    "label": "section",
    "text": "Types of Data"
  },
  {
    "label": "content",
    "text": "The project will generate..."
  }
]
```

### Narrative JSON

```json
{
  "narrative": {
    "template": {
      "title": "Data Management Plan",
      "section": [
        {
          "title": "Types of Data",
          "question": [
            {
              "text": "Types of Data",
              "answer": {
                "json": {
                  "answer": "The project will generate..."
                }
              }
            }
          ]
        }
      ]
    }
  }
}
```

---

## Future Work

1. Compare LLM-based structure detection with the existing rule-based approach.
2. Evaluate structure detection accuracy at the section and subsection level.
3. Measure content preservation using:
   - Word Capture
   - ROUGE-L
   - Word Precision
   - Word Recall
   - Word F1
4. Extend the workflow to populate the complete RDA Common Standard and DMPTool extension schema.
5. Compare multiple open-source models (Llama, Qwen, and Mistral) for DMP narrative structure extraction.

In [1]:
from dmpbridge.llm.llama_client import load_llama

llm = load_llama()

response = llm.invoke(
    "What is a Data Management Plan?"
)

print(response.content)

A Data Management Plan (DMP) is a document that outlines how data will be collected, stored, managed, and preserved throughout the lifecycle of a research project. The purpose of a DMP is to ensure that data are properly managed and made available for future use, reuse, and sharing.

A typical DMP includes information on:

1. **Data collection**: How data will be gathered, including methods, tools, and instruments used.
2. **Data storage**: Where data will be stored (e.g., local servers, cloud storage) and how they will be organized and labeled.
3. **Data security**: Measures to ensure data confidentiality, integrity, and availability, such as access controls, encryption, and backup procedures.
4. **Data sharing**: Plans for sharing data with collaborators, stakeholders, or the public, including any restrictions on use or access.
5. **Metadata management**: How metadata (data about the data) will be created, stored, and managed to facilitate data discovery and reuse.
6. **Data preserva

In [ ]:
from pathlib import Path
import json

from dmpbridge.llm.llama_client import load_llama
from dmpbridge.llm.llm_narrative_blocks import (
    generate_structured_blocks_with_llm,
    save_blocks
)
from dmpbridge.processing.structure_json_builder import (
    build_narrative_json_from_blocks
)
from dmpbridge.utils.file_io import save_json



# Project root


cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    project_root = cwd
else:
    project_root = cwd.parent

print("Project root:", project_root)



# Paths


markdown_path = (
    project_root
    / "data"
    / "pdfplumber_extracted_markdown"
    / "sample1.md"
)

blocks_output_path = (
    project_root
    / "data"
    / "llama_structured_blocks"
    / "sample1_llama_blocks.json"
)

narrative_output_path = (
    project_root
    / "data"
    / "llama_narrative_json"
    / "sample1_llama_narrative.json"
)

print("Markdown exists:", markdown_path.exists())



# Load DMP markdown


dmp_text = markdown_path.read_text(encoding="utf-8")


# Load Ollama Llama


llm = load_llama(
    model_name="llama3.1:8b",
    temperature=0
)



# Step 1: Llama creates structured blocks


structured_blocks = generate_structured_blocks_with_llm(
    llm=llm,
    dmp_text=dmp_text
)

save_blocks(
    blocks=structured_blocks,
    output_path=blocks_output_path
)

print("Saved Llama blocks:", blocks_output_path)
print("Number of blocks:", len(structured_blocks))



# Step 2: Your existing code creates narrative JSON


narrative_json = build_narrative_json_from_blocks(
    structured_blocks=structured_blocks
)

save_json(
    narrative_json,
    narrative_output_path
)

print("Saved narrative JSON:", narrative_output_path)

Project root: c:\Users\Nahid\dmpbridge
Markdown exists: True
Saved Llama blocks: c:\Users\Nahid\dmpbridge\data\llama_structured_blocks\sample1_llama_blocks.json
Number of blocks: 34
Saved narrative JSON: c:\Users\Nahid\dmpbridge\data\llama_narrative_json\sample1_llama_narrative.json
